<a href="https://colab.research.google.com/github/darkfrozen158/09MIAR_04_A_2025-26_Trabajo-Fin-de-M-ster-Inteligencia-Artificial-/blob/main/DEMO_AGENT_AI_BANCARIO_TFM_16_03_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install azure-cosmos
!pip install azure-search-documents
!pip install openai

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.2/114.2 kB 9.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.4/48.4 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 424.9/424.9 kB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 218.2/218.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 307.9/307.9 kB 20.0 MB/s eta 0:00:00


In [3]:
from azure.cosmos import CosmosClient
from azure.search.documents import SearchClient
from azure.core.credentials import AzureKeyCredential
from openai import AzureOpenAI

In [ ]:
COSMOS_ENDPOINT = "https://tu-cosmos.documents.azure.com:443/"
COSMOS_KEY = "COSMOS_KEY"

DATABASE_NAME = "banking-ai"
CONTAINER_NAME = "transactions"

cosmos_client = CosmosClient(COSMOS_ENDPOINT, COSMOS_KEY)

database = cosmos_client.get_database_client(DATABASE_NAME)
container = database.get_container_client(CONTAINER_NAME)

In [ ]:
COSMOS_ENDPOINT = "https://tu-cosmos.documents.azure.com:443/"
COSMOS_KEY = "COSMOS_KEY"

DATABASE_NAME = "banking-ai"
CONTAINER_NAME = "transactions"

cosmos_client = CosmosClient(COSMOS_ENDPOINT, COSMOS_KEY)

database = cosmos_client.get_database_client(DATABASE_NAME)
container = database.get_container_client(CONTAINER_NAME)

In [ ]:
SEARCH_ENDPOINT = "https://tu-search.search.windows.net"
SEARCH_KEY = "SEARCH_KEY"
INDEX_NAME = "banking-documents"

search_client = SearchClient(
    endpoint=SEARCH_ENDPOINT,
    index_name=INDEX_NAME,
    credential=AzureKeyCredential(SEARCH_KEY)
)

In [ ]:
client = AzureOpenAI(
    api_key="OPENAI_KEY",
    api_version="2024-02-15-preview",
    azure_endpoint="https://tu-openai.openai.azure.com/"
)

In [ ]:
def obtener_gastos_cliente(cliente_id, mes):

    query = f"""
    SELECT c.monto, c.fecha, c.categoria
    FROM c
    WHERE c.cliente_id = '{cliente_id}'
    AND STARTSWITH(c.fecha, '{mes}')
    """

    items = list(container.query_items(
        query=query,
        enable_cross_partition_query=True
    ))

    return items

In [ ]:
def buscar_documentacion(pregunta):

    results = search_client.search(
        search_text=pregunta,
        top=3
    )

    documentos = []

    for r in results:
        documentos.append(r["content"])

    return documentos

In [ ]:
def construir_contexto(transacciones, documentos):

    gasto_total = sum(t["monto"] for t in transacciones)

    contexto = f"""
    El cliente gastó {gasto_total} en el mes consultado.

    Transacciones:
    {transacciones}

    Documentación del banco:
    {documentos}
    """

    return contexto

In [ ]:
def generar_respuesta(pregunta, contexto):

    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {
                "role": "system",
                "content": "Eres un asistente bancario que responde de forma clara y segura."
            },
            {
                "role": "user",
                "content": f"""
                Pregunta del cliente:
                {pregunta}

                Contexto:
                {contexto}
                """
            }
        ]
    )

    return response.choices[0].message.content

In [ ]:
def ai_agent_responder(cliente_id, pregunta):

    # obtener datos operativos
    transacciones = obtener_gastos_cliente(cliente_id, "2026-01")

    # obtener documentación RAG
    docs = buscar_documentacion(pregunta)

    # construir contexto
    contexto = construir_contexto(transacciones, docs)

    # generar respuesta
    respuesta = generar_respuesta(pregunta, contexto)

    return respuesta

In [ ]:
respuesta = ai_agent_responder(
    cliente_id="12345",
    pregunta="¿Cuánto gasté en enero?"
)

print(respuesta)